# Load and play with Huggingface Dataset 
### Learn manipulation and create finetuning dataset for SLM finetuning

In [10]:
from datasets import load_dataset

In [11]:
# Load JDhruv14/Bhagavad-Gita-QA dataset from Hugging Face
dataset_1 = load_dataset("JDhruv14/Bhagavad-Gita-QA")

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 420aaf1d-5c96-40d3-ac81-db03cd7c9e91)')' thrown while requesting HEAD https://huggingface.co/datasets/JDhruv14/Bhagavad-Gita-QA/resolve/main/README.md
Retrying in 1s [Retry 1/5].


In [12]:
# studay dataset structure
print(dataset_1)

DatasetDict({
    train: Dataset({
        features: ['chapter_no', 'verse_no', 'question', 'answer'],
        num_rows: 3500
    })
})


In [13]:
print(dataset_1['train'][0])

{'chapter_no': 1, 'verse_no': 1, 'question': 'Why does Dhritarashtra ask Sanjaya to describe the battlefield?', 'answer': 'Dhritarashtra is blind, both physically and symbolically — representing ignorance. He wants Sanjaya to narrate what is happening because he is anxious about the outcome of the war and whether his sons (Kauravas) will win.'}


In [14]:
data_sample = dataset_1['train'].shuffle(seed=42).select(range(5))
data_sample[:3]

{'chapter_no': [2, 7, 17],
 'verse_no': [57, 23, 7],
 'question': ['Why is it important for a person to not rejoice in good or recoil from evil according to this verse?',
  "How does Krishna describe people with 'meager intelligence' in relation to their worship practices in this verse?",
  "What can we learn about the nature of human actions and preferences from Krishna's statement that food, sacrifice, penance, and charity are all of three kinds?"],
 'answer': ['It is important because reacting strongly to pleasure or pain creates attachment and aversion, which disturb the mind. By not rejoicing in good or recoiling from evil, a person can maintain a steady mind, free from the turmoil caused by changing circumstances. This steadiness is essential for spiritual growth and inner peace.',
  "Krishna refers to those with 'meager intelligence' as individuals who seek temporary outcomes by worshipping various gods for material gains. He suggests that their understanding is limited, as they

In [15]:
dataset_2 = load_dataset("serpentilec137/gita-verse-qna-dataset")    

In [16]:
dataset_2

DatasetDict({
    train: Dataset({
        features: ['question', 'answer', 'source_chapter', 'source_verse'],
        num_rows: 1138
    })
    test: Dataset({
        features: ['question', 'answer', 'source_chapter', 'source_verse'],
        num_rows: 127
    })
})

In [17]:
data_sample = dataset_2['train'].shuffle(seed=42).select(range(5))
data_sample[:3]

{'question': ['Was the universal form shown to Arjuna the same as those manifested at other times, like to Duryodhana, according to the purport of Gita 11.47?',
  'What does a person in material consciousness not know, and what is the symptom of their nescience, as per the purport of Gita 3.27?',
  'How does a conditioned soul sometimes get relieved from repetitive so-called happiness, and what is the nature of pursuing self-realization, as per the purport of Gita 18.36-37?'],
 'answer': ["The purport states Arjuna wanted universal form; Krsna showed it by His mercy. Form glaring like sun, many faces rapidly changing. Manifested by Krsna's internal potency, inconceivable by human speculation. No one saw this universal form before Arjuna. Because shown to Arjuna, other devotees in heavenly/other planets could also see. Comment: form shown to Duryodhana during peace negotiations different from this one shown to Arjuna. Clearly stated no one saw this form before.",
  "The purport explains

In [19]:
# Process datastes to json file
import json
import os
OUTPUT_DIR = "data/processed"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "gita_qna_for_finetune.jsonl")

# Create jsonl file with each line  formatted for using with unsloth library for finetuning with llama3.1 template
# format of each line:
# {"messages": [
#   {"role": "system", "content": "You are a wise teacher drawing from Bhagavad Gita."},
#   {"role": "user", "content": "Using Chapter 2, Verse 47, answer: What is true action?"},
#   {"role": "assistant", "content": "Karmany evadhikaraste... (your Gita-based answer)"}
# ]}


# dataset 1 keys - 'chapter_no', 'verse_no', 'question', 'answer'
processed_dataset = []
for item in dataset_1['train']:
    processed_dataset.append({
        "messages": [
            {"role": "system", "content": "You are a wise teacher drawing from Bhagavad Gita."},
            {"role": "user", "content": f"Using Chapter {item['chapter_no']}, Verse {item['verse_no']}, answer: {item['question']}"},
            {"role": "assistant", "content": item['answer']}
        ]
    })

# dataset 2 keys - 'question', 'answer', 'source_chapter', 'source_verse'
for item in dataset_2['train']:
    processed_dataset.append({
        "messages": [
            {"role": "system", "content": "You are a wise teacher drawing from Bhagavad Gita."},
            {"role": "user", "content": f"Using Chapter {item['source_chapter']}, Verse {item['source_verse']}, answer: {item['question']}"},
            {"role": "assistant", "content": item['answer']}
        ]
    })

os.makedirs(OUTPUT_DIR, exist_ok=True)

#  Save to JSONL
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    for ex in processed_dataset:
        f.write(json.dumps(ex, ensure_ascii=False) + '\n')

print(f"Processed dataset saved to {OUTPUT_FILE}")

Processed dataset saved to data/processed/gita_qna_for_finetune.jsonl
